In [1]:
import os
import torch
import numpy as np
import pandas as pd
from typing import Dict, List
import time
from dotenv import load_dotenv

from langchain_core.messages import ChatMessage
from langchain_core.prompts import ChatMessagePromptTemplate
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_text_splitters.base import TextSplitter

from financerag.common import get_query_and_retrieved_corpus_text, process_retrieval_df, get_final_result 
from financerag.task import *
from financerag.retrieval import DenseRetrieval
from financerag.hipporag import HippoRAG
from financerag.retrieval import BM25, BM25_Retriever
from financerag.rerank import CrossEncoderReranker

from sentence_transformers import CrossEncoder
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

/Users/mac/Desktop/Code/Personal_Project/DeepLearningProject/RAG_Project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(".env")
GG_API_KEY = os.environ.get('GOOGLE_API_KEY')
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')

In [5]:
print(f"FinQABench task")
finqabench_task = FinQABenchTask()
finqabench_task.load(corpus_path="finance_dataset")
len(finqabench_task.corpus)

FinQABench task


92

In [17]:
# Set up vector database

import time
import os
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "ragfinance"  # change if desired

existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=768,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        deletion_protection="enabled",  # Defaults to "disabled"
    )
    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)

index = pc.Index(index_name)
vector_store = PineconeVectorStore(
    index=index, embedding=GoogleGenerativeAIEmbeddings(model="models/embedding-001")
)

In [12]:
results = vector_store.similarity_search_with_score("Google", k = 5)
results[0][0]


Document(id='f1a00c8d-b9df-46ae-b398-44d1e6582bbe', metadata={'triple': ['Alice', 'works at', 'Google']}, page_content='Alice works at Google')

In [3]:
def load_query(dataset_name : str, new_path_to_query = None) -> Dict[str, str]:
    query_df = pd.read_json(f"finance_dataset/{dataset_name.lower()}_queries.jsonl/queries.jsonl", lines = True)
    # Convert into Dict[str, str]
    query_dict = {row['_id'] : row['text'] for i, row in query_df.iterrows()}
    return query_dict

    
def load_corpus(dataset_name : str, new_path_to_corpus = None) -> Dict[str, str]:
    corpus_df = pd.read_json(f"finance_dataset/{dataset_name.lower()}_corpus.jsonl/corpus.jsonl", lines = True)
    # Convert into Dict[str, str]
    corpus_dict = {row['_id'] : row['text'] for i, row in corpus_df.iterrows()}
    return corpus_dict

In [4]:
# Set up vector database
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
embeddings = GoogleGenerativeAIEmbeddings(model = "models/text-embedding-004", request_options = {'timeout' : 100000})
index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [ ]:
def load_corpus(dataset_name: str, corpus_path: str = None):
    """This function is used to load corpus data from the given dataset name

    Args:
        corpus_path (str, optional): Root path to corpus. Defaults to None.

    Returns:
        corpus (Dict[str,str]) : Corpus dictionary
    """
    if corpus_path is None:
        corpus_path = "finance_dataset"
    corpus_df = pd.read_json(
        f"{corpus_path}/{dataset_name}_corpus.jsonl/corpus.jsonl", lines=True
    )
    # Filter rows with the same id and without text
    corpus_df.drop_duplicates(inplace=True)
    corpus_df["text_concat"] = corpus_df["text"]
    if "title" in corpus_df.columns:
        corpus_df["text_concat"] = corpus_df["title"] + " " + corpus_df["text"]
        corpus_df["text_concat"] = corpus_df["text_concat"].str.strip()
    corpus_df["text_len"] = corpus_df["text_concat"].str.len()
    # Extract the row with text greater than 0
    corpus_df = corpus_df.loc[corpus_df["text_len"] > 0]
    corpus = {row["_id"]: row["text_concat"] for _, row in corpus_df.iterrows()}
    return corpus

corpus = load_corpus('')

In [28]:
def get_vector_store():
    embeddings = GoogleGenerativeAIEmbeddings(model = "models/text-embedding-004",  request_options = {'timeout' : 100000})
    index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))
    
    vector_store = FAISS(
        embedding_function=embeddings,
        index=index,
        docstore=InMemoryDocstore(),
        index_to_docstore_id={},
    )
    return vector_store

In [29]:
if torch.backends.mps.is_available():
    mps_device = torch.device("mps")

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
device

'mps'

In [3]:
# Get dataset name first
import os
dataset_names = []
for f in os.listdir('finance_dataset'):
    if f.endswith("tsv"):
       dataset_names.append(f.split('_')[0])
dataset_names  

['MultiHeirtt',
 'FinQA',
 'FinanceBench',
 'ConvFinQA',
 'FinQABench',
 'TATQA',
 'FinDER']

In [8]:
print(f"ConvFinQA task")
convfinqa_task = ConvFinQATask()
convfinqa_task.load(corpus_path="finance_dataset")
len(convfinqa_task.queries)

ConvFinQA task


421

In [31]:
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 300
text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

In [9]:
model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

In [10]:
for dataset_name in dataset_names:
    task_variable = f"{dataset_name.lower()}_task"
    script_string = f"""
    # {dataset_name} Task
    print(f"{dataset_name} task")
    {task_variable} = {dataset_name}Task()
    {task_variable}.load(corpus_path='finance_subset_dataset')
    vector_store = get_vector_store()
    {dataset_name}_hippo_rag = HippoRAG(retriever = vector_store)
    {dataset_name}_hippo_rag.offline_indexing(corpus = {task_variable}.corpus, batch_size = 4)
    {task_variable}_result = {dataset_name}_hippo_rag.retrieve(queries = {task_variable}.queries, corpus = {task_variable}.corpus, top_k = 10)
    # {task_variable}_reranker = CrossEncoderReranker(queries = {task_variable}.queries, corpus = {task_variable}.corpus, reranker = model)
    # {task_variable}_final_result = {task_variable}_reranker.rerank(retrieved_result = {task_variable}_result, top_k = 10)
    {task_variable}.save_retrieved_results({task_variable}_result, method_name = 'hippo_rag_baseline')
    """
    print(script_string)


    # MultiHeirtt Task
    print(f"MultiHeirtt task")
    multiheirtt_task = MultiHeirttTask()
    multiheirtt_task.load(corpus_path='finance_subset_dataset')
    vector_store = get_vector_store()
    MultiHeirtt_hippo_rag = HippoRAG(retriever = vector_store)
    MultiHeirtt_hippo_rag.offline_indexing(corpus = multiheirtt_task.corpus, batch_size = 4)
    multiheirtt_task_result = MultiHeirtt_hippo_rag.retrieve(queries = multiheirtt_task.queries, corpus = multiheirtt_task.corpus, top_k = 10)
    # multiheirtt_task_reranker = CrossEncoderReranker(queries = multiheirtt_task.queries, corpus = multiheirtt_task.corpus, reranker = model)
    # multiheirtt_task_final_result = multiheirtt_task_reranker.rerank(retrieved_result = multiheirtt_task_result, top_k = 10)
    multiheirtt_task.save_retrieved_results(multiheirtt_task_result, method_name = 'hippo_rag_baseline')
    

    # FinQA Task
    print(f"FinQA task")
    finqa_task = FinQATask()
    finqa_task.load(corpus_path='finance_subset_dat

In [32]:
# MultiHeirtt Task
print(f"MultiHeirtt task")
multiheirtt_task = MultiHeirttTask()
multiheirtt_task.load(corpus_path='finance_subset_dataset')
vector_store = get_vector_store()
MultiHeirtt_hippo_rag = HippoRAG(retriever = vector_store)
MultiHeirtt_hippo_rag.offline_indexing(corpus = multiheirtt_task.corpus, batch_size = 4)
multiheirtt_task_result = MultiHeirtt_hippo_rag.retrieve(queries = multiheirtt_task.queries, corpus = multiheirtt_task.corpus, top_k = 10)
# multiheirtt_task_reranker = CrossEncoderReranker(queries = multiheirtt_task.queries, corpus = multiheirtt_task.corpus, reranker = model)
# multiheirtt_task_final_result = multiheirtt_task_reranker.rerank(retrieved_result = multiheirtt_task_result, top_k = 10)
multiheirtt_task.save_retrieved_results(multiheirtt_task_result, method_name = 'hippo_rag_baseline')


# FinQA Task
print(f"FinQA task")
finqa_task = FinQATask()
finqa_task.load(corpus_path='finance_subset_dataset')
vector_store = get_vector_store()
FinQA_hippo_rag = HippoRAG(retriever = vector_store)
FinQA_hippo_rag.offline_indexing(corpus = finqa_task.corpus, batch_size = 4)
finqa_task_result = FinQA_hippo_rag.retrieve(queries = finqa_task.queries, corpus = finqa_task.corpus, top_k = 10)
# finqa_task_reranker = CrossEncoderReranker(queries = finqa_task.queries, corpus = finqa_task.corpus, reranker = model)
# finqa_task_final_result = finqa_task_reranker.rerank(retrieved_result = finqa_task_result, top_k = 10)
finqa_task.save_retrieved_results(finqa_task_result, method_name = 'hippo_rag_baseline')


# FinanceBench Task
print(f"FinanceBench task")
financebench_task = FinanceBenchTask()
financebench_task.load(corpus_path='finance_subset_dataset')
vector_store = get_vector_store()
FinanceBench_hippo_rag = HippoRAG(retriever = vector_store)
FinanceBench_hippo_rag.offline_indexing(corpus = financebench_task.corpus, batch_size = 4)
financebench_task_result = FinanceBench_hippo_rag.retrieve(queries = financebench_task.queries, corpus = financebench_task.corpus, top_k = 10)
# financebench_task_reranker = CrossEncoderReranker(queries = financebench_task.queries, corpus = financebench_task.corpus, reranker = model)
# financebench_task_final_result = financebench_task_reranker.rerank(retrieved_result = financebench_task_result, top_k = 10)
financebench_task.save_retrieved_results(financebench_task_result, method_name = 'hippo_rag_baseline')


# ConvFinQA Task
print(f"ConvFinQA task")
convfinqa_task = ConvFinQATask()
convfinqa_task.load(corpus_path='finance_subset_dataset')
vector_store = get_vector_store()
ConvFinQA_hippo_rag = HippoRAG(retriever = vector_store)
ConvFinQA_hippo_rag.offline_indexing(corpus = convfinqa_task.corpus, batch_size = 4)
convfinqa_task_result = ConvFinQA_hippo_rag.retrieve(queries = convfinqa_task.queries, corpus = convfinqa_task.corpus, top_k = 10)
# convfinqa_task_reranker = CrossEncoderReranker(queries = convfinqa_task.queries, corpus = convfinqa_task.corpus, reranker = model)
# convfinqa_task_final_result = convfinqa_task_reranker.rerank(retrieved_result = convfinqa_task_result, top_k = 10)
convfinqa_task.save_retrieved_results(convfinqa_task_result, method_name = 'hippo_rag_baseline')


# FinQABench Task
print(f"FinQABench task")
finqabench_task = FinQABenchTask()
finqabench_task.load(corpus_path='finance_subset_dataset')
vector_store = get_vector_store()
FinQABench_hippo_rag = HippoRAG(retriever = vector_store)
FinQABench_hippo_rag.offline_indexing(corpus = finqabench_task.corpus, batch_size = 4)
finqabench_task_result = FinQABench_hippo_rag.retrieve(queries = finqabench_task.queries, corpus = finqabench_task.corpus, top_k = 10)
# finqabench_task_reranker = CrossEncoderReranker(queries = finqabench_task.queries, corpus = finqabench_task.corpus, reranker = model)
# finqabench_task_final_result = finqabench_task_reranker.rerank(retrieved_result = finqabench_task_result, top_k = 10)
finqabench_task.save_retrieved_results(finqabench_task_result, method_name = 'hippo_rag_baseline')


# TATQA Task
print(f"TATQA task")
tatqa_task = TATQATask()
tatqa_task.load(corpus_path='finance_subset_dataset')
vector_store = get_vector_store()
TATQA_hippo_rag = HippoRAG(retriever = vector_store)
TATQA_hippo_rag.offline_indexing(corpus = tatqa_task.corpus, batch_size = 4)
tatqa_task_result = TATQA_hippo_rag.retrieve(queries = tatqa_task.queries, corpus = tatqa_task.corpus, top_k = 10)
# tatqa_task_reranker = CrossEncoderReranker(queries = tatqa_task.queries, corpus = tatqa_task.corpus, reranker = model)
# tatqa_task_final_result = tatqa_task_reranker.rerank(retrieved_result = tatqa_task_result, top_k = 10)
tatqa_task.save_retrieved_results(tatqa_task_result, method_name = 'hippo_rag_baseline')


# FinDER Task
print(f"FinDER task")
finder_task = FinDERTask()
finder_task.load(corpus_path='finance_subset_dataset')
vector_store = get_vector_store()
FinDER_hippo_rag = HippoRAG(retriever = vector_store)
FinDER_hippo_rag.offline_indexing(corpus = finder_task.corpus, batch_size = 4)
finder_task_result = FinDER_hippo_rag.retrieve(queries = finder_task.queries, corpus = finder_task.corpus, top_k = 10)
# finder_task_reranker = CrossEncoderReranker(queries = finder_task.queries, corpus = finder_task.corpus, reranker = model)
# finder_task_final_result = finder_task_reranker.rerank(retrieved_result = finder_task_result, top_k = 10)
finder_task.save_retrieved_results(finder_task_result, method_name = 'hippo_rag_baseline')

MultiHeirtt task


Retrieving:  11%|█         | 106/974 [01:40<13:43,  1.05it/s]


KeyboardInterrupt: 

In [75]:
from optimum.quanto import quantize, freeze, qint8

In [ ]:
from langchain_core.output_parsers import ListOutputParser
import re
class CustomizedListParser(ListOutputParser):
    def parse(self, text) -> List[List[List[str]]]:
        pattern = r"\[\s*(?:\[\s*(?:\[[^\]]*?\]\s*,?\s*\n*)*\s*\]\s*,?\s*)*\s*\]"
        result = re.findall(pattern, text)
        if result:
            extracted_text = re.findall(pattern, text)[0]
            return eval(extracted_text)
        return []

In [76]:
# Load model directly
from transformers import AutoModelForCausalLM, AutoTokenizer
model_id = "llmware/slim-summary"
model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code = True)
tokenizer = AutoTokenizer.from_pretrained(model_id)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [78]:
quantize(model, weights= qint8)

In [80]:
# finalize quantized weights for our model
freeze(model)

In [81]:
print(f"After quantization model memory: {model.get_memory_footprint() / 10**10}GB")

After quantization model memory: 1.12027474GB


In [82]:
corpus = []
for corpus_text in financebench_task.corpus.values():
    if (len(corpus_text) > 2000):
        corpus.append(corpus_text)

corpus = corpus[:10]


In [ ]:
def clean_memory():
    import gc
    gc.collect()
    torch.mps.empty_cache()

In [84]:

function = "summarize"
params = "key points (1)"

prompts = ["<human>: " + corpus + "\n" + f"<{function}> {params} </{function}>\n<bot>:" for corpus in corpus]


tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer(prompts, return_tensors="pt", padding = True)
start_of_input = len(inputs.input_ids[0])

outputs = model.generate(
    input_ids = inputs['input_ids'],
    attention_mask = inputs['attention_mask'],
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=False,
    temperature=0.3,
    max_new_tokens=50
)

output_only = tokenizer.batch_decode(outputs)

# print("output only: ", output_only)  


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


In [50]:
inputs['input_ids'].shape

torch.Size([10, 1100])

In [39]:
pipe = pipeline(task = 'summarization', model = model, tokenizer=tokenizer, device = device, )

The model 'StableLMEpochForCausalLM' is not supported for summarization. Supported models are ['BartForConditionalGeneration', 'BigBirdPegasusForConditionalGeneration', 'BlenderbotForConditionalGeneration', 'BlenderbotSmallForConditionalGeneration', 'EncoderDecoderModel', 'FSMTForConditionalGeneration', 'GPTSanJapaneseForConditionalGeneration', 'LEDForConditionalGeneration', 'LongT5ForConditionalGeneration', 'M2M100ForConditionalGeneration', 'MarianMTModel', 'MBartForConditionalGeneration', 'MT5ForConditionalGeneration', 'MvpForConditionalGeneration', 'NllbMoeForConditionalGeneration', 'PegasusForConditionalGeneration', 'PegasusXForConditionalGeneration', 'PLBartForConditionalGeneration', 'ProphetNetForConditionalGeneration', 'SeamlessM4TForTextToText', 'SeamlessM4Tv2ForTextToText', 'SwitchTransformersForConditionalGeneration', 'T5ForConditionalGeneration', 'UMT5ForConditionalGeneration', 'XLMProphetNetForConditionalGeneration'].


In [ ]:
generate_kwargs = {
    "do_sample": False,
    "temperature": 0,
    "max_new_tokens": 35,
}

In [ ]:
# us-east-1c

In [ ]:
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 300
text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

for corpus_id, corpus_text in financebench_task.corpus.items():
    # Use a tokenizer here to check the lpass
    pass

In [ ]:
for dataset_name in dataset_names:
    task_variable = f"{dataset_name.lower()}_task"
    script_string = f"""
    # {dataset_name} Task
    {task_variable}_final_result = extract_result({task_variable}_result, k = 10)
    {task_variable}.save_retrieved_results({task_variable}_final_result, method_name = 'dense_retrieval_split_only')
    """
    print(script_string)

In [ ]:
def extract_result(task_result : Dict[str, Dict[str, float]], k = 10):
    final_result = {}
    for query_id, doc_dict in task_result.items():
        final_result[query_id] = {}
        for i, (corpus_id, score) in enumerate(doc_dict.items()):
            if (i == k): break
            final_result[query_id][corpus_id] = score
    return final_result

In [ ]:

# # FinQA Task
# finqa_task_final_result = extract_result(finqa_task_result, k = 10)
# finqa_task.save_retrieved_results(finqa_task_final_result, method_name = 'dense_retrieval_split_only')


# # FinanceBench Task
# financebench_task_final_result = extract_result(financebench_task_result, k = 10)
# financebench_task.save_retrieved_results(financebench_task_final_result, method_name = 'dense_retrieval_split_only')


# # ConvFinQA Task
# convfinqa_task_final_result = extract_result(convfinqa_task_result, k = 10)
# convfinqa_task.save_retrieved_results(convfinqa_task_final_result, method_name = 'dense_retrieval_split_only')


# # FinQABench Task
# finqabench_task_final_result = extract_result(finqabench_task_result, k = 10)
# finqabench_task.save_retrieved_results(finqabench_task_final_result, method_name = 'dense_retrieval_split_only')


# # TATQA Task
# tatqa_task_final_result = extract_result(tatqa_task_result, k = 10)
# tatqa_task.save_retrieved_results(tatqa_task_final_result, method_name = 'dense_retrieval_split_only')


# FinDER Task
finder_task_final_result = extract_result(finder_task_result, k = 10)
finder_task.save_retrieved_results(finder_task_final_result, method_name = 'dense_retrieval_split_only')

In [ ]:





# TATQA Task
tatqa_task_final_result = extract_result(tatqa_task_result, k = 10)
tatqa_task.save_retrieved_results(tatqa_task_final_result, method_name = 'dense_retrieval_only')


# FinDER Task
finder_task_final_result = extract_result(finder_task_result, k = 10)
finder_task.save_retrieved_results(finder_task_final_result, method_name = 'dense_retrieval_only')

In [ ]:
method_name = 'dense_retrieval_split_document_and_reranking'
final_result = get_final_result(dataset_names, method_name = method_name)

In [ ]:
final_result

In [ ]:
final_result

In [ ]:
final_result.to_csv(f'submission_{method_name}.csv', index = False)

In [ ]:
import torch.nn as nn

In [ ]:
# Implement HippoRAG version 2 to see the result, original HippoRAG threw away context that can be useful for retrieving information

In [ ]:

# FinQA Task
print(f"FinQA task")
finqa_task = FinQATask()
finqa_task.load()
vector_store = get_vector_store()
finqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finqa_task.metadata.dataset_name)
finqa_task_retriever.load_corpus_without_splitting(corpus = finqa_task.corpus, saved_index = False)
finqa_task_result = finqa_task.retrieve(retriever = finqa_task_retriever, top_k = 50)
finqa_task_reranker = CrossEncoderReranker(queries = finqa_task.queries, corpus = finqa_task.corpus, reranker = model)
finqa_task_final_result = finqa_task_reranker.rerank(finqa_task_result, top_k = 10)
finqa_task.save_retrieved_results(finqa_task_final_result, method_name = 'dense_retrieval_and_reranking')





In [ ]:
# # MultiHeirtt Task
# multiheirtt_task_final_result = extract_result(multiheirtt_task_result, k = 10)
# multiheirtt_task.save_retrieved_results(multiheirtt_task_final_result, method_name = 'dense_retrieval_only')


# FinQA Task
finqa_task_final_result = extract_result(finqa_task_result, k = 10)
finqa_task.save_retrieved_results(finqa_task_final_result, method_name = 'dense_retrieval_only')





In [7]:
# For each dataset, randomly picking 500 samples within it and save the data in the correct format 
# Get dataset name first
import os
dataset_names = []
MIN_SAMPLES = 1000
for f in os.listdir('finance_dataset'):
    if f.endswith("tsv"):
        dataset_name = f.split("_")[0].lower()
        # read corpus
        corpus_df = pd.read_json(f"finance_dataset/{dataset_name}_corpus.jsonl/corpus.jsonl", lines = True)

        n_samples = min(MIN_SAMPLES, len(corpus_df))
        # Pick index randomly
        selected_indices = np.random.choice(np.arange(len(corpus_df)), size = n_samples, replace = False)

        subset_df = corpus_df.iloc[selected_indices]
        # Write file
        if (not os.path.exists('finance_subset_dataset')):
            os.mkdir('finance_subset_dataset')
        if (not os.path.exists(f'finance_subset_dataset/{dataset_name}_corpus.jsonl')):
            os.mkdir(f"finance_subset_dataset/{dataset_name}_corpus.jsonl")
        with open(f"finance_subset_dataset/{dataset_name}_corpus.jsonl/corpus.jsonl", "w") as f:
            f.write(subset_df.to_json(orient="records", lines=True, force_ascii=False))
        


In [ ]:
# 